In [ ]:
# Install liboqs from source
!apt-get install -y cmake ninja-build libssl-dev 2>/dev/null | tail -5
!pip install pytest pytest-xdist pyyaml 2>/dev/null | tail -3

!git clone --depth 1 https://github.com/open-quantum-safe/liboqs.git
!cmake -S liboqs -B liboqs/build -DBUILD_SHARED_LIBS=ON -DOQS_BUILD_ONLY_LIB=ON -GNinja
!cmake --build liboqs/build --parallel 4
!cmake --install liboqs/build

!git clone --depth 1 https://github.com/open-quantum-safe/liboqs-python.git
!pip install ./liboqs-python

print("liboqs installed successfully!")

Reading state information...
ninja-build is already the newest version (1.10.1-1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
libssl-dev is already the newest version (3.0.2-0ubuntu1.23).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
fatal: destination path 'liboqs' already exists and is not an empty directory.
-- OQS Testing: Pytest parallel workers set to 'auto'
-- CMAKE_BUILD_TYPE=Release
-- Alg enablement unchanged
-- Configuring done (0.3s)
-- Generating done (0.9s)
-- Build files have been written to: /content/liboqs/build
[1737/1737] Creating library symlink lib/liboqs.so.9 lib/liboqs.so
-- Install configuration: "Release"
-- Up-to-date: /usr/local/lib/cmake/liboqs/liboqsConfig.cmake
-- Up-to-date: /usr/local/lib/cmake/liboqs/liboqsConfigVersion.cmake
-- Up-to-date: /usr/local/lib/pkgconfig/liboqs.pc
-- Installing: /usr/local/lib/liboqs.so.0.16.0
-- Up-to-date: /usr/local/lib/liboqs.so.9
-- Up-to-date: /usr/local/lib/liboqs.so
-- Up-to-date: 

In [ ]:
slh_map = {
    "SPHINCS+-SHA2-128s-simple": "SLH-DSA-SHA2-128s",
    "SPHINCS+-SHA2-192s-simple": "SLH-DSA-SHA2-192s",
    "SPHINCS+-SHA2-256s-simple": "SLH-DSA-SHA2-256s"
}


import oqs
available = oqs.get_enabled_sig_mechanisms()
sphincs_available = [x for x in available if "SPHINCS" in x or "SLH" in x]
print("Available SPHINCS/SLH names:", sphincs_available)

liboqs-python faulthandler is disabled


INFO:oqs.oqs:liboqs-python faulthandler is disabled


Available SPHINCS/SLH names: ['SLH_DSA_PURE_SHA2_128S', 'SLH_DSA_PURE_SHA2_128F', 'SLH_DSA_PURE_SHA2_192S', 'SLH_DSA_PURE_SHA2_192F', 'SLH_DSA_PURE_SHA2_256S', 'SLH_DSA_PURE_SHA2_256F', 'SLH_DSA_PURE_SHAKE_128S', 'SLH_DSA_PURE_SHAKE_128F', 'SLH_DSA_PURE_SHAKE_192S', 'SLH_DSA_PURE_SHAKE_192F', 'SLH_DSA_PURE_SHAKE_256S', 'SLH_DSA_PURE_SHAKE_256F', 'SLH_DSA_SHA2_224_PREHASH_SHA2_128S', 'SLH_DSA_SHA2_224_PREHASH_SHA2_128F', 'SLH_DSA_SHA2_224_PREHASH_SHA2_192S', 'SLH_DSA_SHA2_224_PREHASH_SHA2_192F', 'SLH_DSA_SHA2_224_PREHASH_SHA2_256S', 'SLH_DSA_SHA2_224_PREHASH_SHA2_256F', 'SLH_DSA_SHA2_224_PREHASH_SHAKE_128S', 'SLH_DSA_SHA2_224_PREHASH_SHAKE_128F', 'SLH_DSA_SHA2_224_PREHASH_SHAKE_192S', 'SLH_DSA_SHA2_224_PREHASH_SHAKE_192F', 'SLH_DSA_SHA2_224_PREHASH_SHAKE_256S', 'SLH_DSA_SHA2_224_PREHASH_SHAKE_256F', 'SLH_DSA_SHA2_256_PREHASH_SHA2_128S', 'SLH_DSA_SHA2_256_PREHASH_SHA2_128F', 'SLH_DSA_SHA2_256_PREHASH_SHA2_192S', 'SLH_DSA_SHA2_256_PREHASH_SHA2_192F', 'SLH_DSA_SHA2_256_PREHASH_SHA2_256S', 

In [ ]:
import oqs
kems = [x for x in oqs.get_enabled_kem_mechanisms() if "KEM" in x]
sigs_dsa = [x for x in oqs.get_enabled_sig_mechanisms() if "DSA" in x and "SLH" not in x]
print("ML-KEM names:", kems[:5])
print("ML-DSA names:", sigs_dsa[:5])

In [ ]:
# Run benchmarks
import time
import statistics
import json
from cryptography.hazmat.primitives.asymmetric import rsa, ec, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.backends import default_backend

ITERATIONS = 1000
message = b"Benchmark message for signing"

def benchmark(fn, iterations=ITERATIONS):
    times = []
    result = None
    for _ in range(iterations):
        t0 = time.perf_counter()
        result = fn()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)
    return round(statistics.mean(times), 4), round(statistics.stdev(times), 4), result

results = {}

# ML-KEM (FIPS 203)
for alg in ["ML-KEM-512", "ML-KEM-768", "ML-KEM-1024"]:
    print(f"Benchmarking {alg}...")
    kem = oqs.KeyEncapsulation(alg)
    kg_m, kg_s, pub = benchmark(lambda: kem.generate_keypair())
    en_m, en_s, (ct, ss) = benchmark(lambda: kem.encap_secret(pub))
    de_m, de_s, _ = benchmark(lambda: kem.decap_secret(ct))
    results[alg] = {
        "type": "KEM",
        "keygen_ms": kg_m, "keygen_std": kg_s,
        "op1_label": "encap_ms",  "op1_ms": en_m, "op1_std": en_s,
        "op2_label": "decap_ms",  "op2_ms": de_m, "op2_std": de_s,
        "pub_key_bytes": len(pub), "ct_sig_bytes": len(ct)
    }
    kem.free()

# ML-DSA (FIPS 204)
for alg in ["ML-DSA-44", "ML-DSA-65", "ML-DSA-87"]:
    print(f"Benchmarking {alg}...")
    sig = oqs.Signature(alg)
    kg_m, kg_s, pub = benchmark(lambda: sig.generate_keypair())
    sn_m, sn_s, signature = benchmark(lambda: sig.sign(message))
    vr_m, vr_s, _ = benchmark(lambda: sig.verify(message, signature, pub))
    results[alg] = {
        "type": "SIG",
        "keygen_ms": kg_m, "keygen_std": kg_s,
        "op1_label": "sign_ms",   "op1_ms": sn_m, "op1_std": sn_s,
        "op2_label": "verify_ms", "op2_ms": vr_m, "op2_std": vr_s,
        "pub_key_bytes": len(pub), "ct_sig_bytes": len(signature)
    }
    sig.free()

# SLH-DSA (FIPS 205) — uses SPHINCS+ name in liboqs
slh_map = {
    "SLH_DSA_PURE_SHA2_128S": "SLH-DSA-SHA2-128s",
    "SLH_DSA_PURE_SHA2_192S": "SLH-DSA-SHA2-192s",
    "SLH_DSA_PURE_SHA2_256S": "SLH-DSA-SHA2-256s"
}
for oqs_name, label in slh_map.items():
    print(f"Benchmarking {label} (slow — ~5 min)...")
    sig = oqs.Signature(oqs_name)
    kg_m, kg_s, pub = benchmark(lambda: sig.generate_keypair())
    sn_m, sn_s, signature = benchmark(lambda: sig.sign(message))
    vr_m, vr_s, _ = benchmark(lambda: sig.verify(message, signature, pub))
    results[label] = {
        "type": "SIG",
        "keygen_ms": kg_m, "keygen_std": kg_s,
        "op1_label": "sign_ms",   "op1_ms": sn_m, "op1_std": sn_s,
        "op2_label": "verify_ms", "op2_ms": vr_m, "op2_std": vr_s,
        "pub_key_bytes": len(pub), "ct_sig_bytes": len(signature)
    }
    sig.free()

# RSA-2048 (classical)
print("Benchmarking RSA-2048...")
kg_m, kg_s, rsa_priv = benchmark(
    lambda: rsa.generate_private_key(public_exponent=65537, key_size=2048, backend=default_backend()))
rsa_pub = rsa_priv.public_key()
pub_bytes = rsa_pub.public_bytes(serialization.Encoding.DER, serialization.PublicFormat.SubjectPublicKeyInfo)
sn_m, sn_s, rsa_sig = benchmark(lambda: rsa_priv.sign(message, padding.PKCS1v15(), hashes.SHA256()))
vr_m, vr_s, _ = benchmark(lambda: rsa_pub.verify(rsa_sig, message, padding.PKCS1v15(), hashes.SHA256()))
results["RSA-2048"] = {
    "type": "SIG",
    "keygen_ms": kg_m, "keygen_std": kg_s,
    "op1_label": "sign_ms",   "op1_ms": sn_m, "op1_std": sn_s,
    "op2_label": "verify_ms", "op2_ms": vr_m, "op2_std": vr_s,
    "pub_key_bytes": len(pub_bytes), "ct_sig_bytes": len(rsa_sig)
}

# ECDSA P-256 (classical)
print("Benchmarking ECDSA P-256...")
kg_m, kg_s, ec_priv = benchmark(
    lambda: ec.generate_private_key(ec.SECP256R1(), default_backend()))
ec_pub = ec_priv.public_key()
ec_pub_bytes = ec_pub.public_bytes(serialization.Encoding.DER, serialization.PublicFormat.SubjectPublicKeyInfo)
sn_m, sn_s, ec_sig = benchmark(lambda: ec_priv.sign(message, ec.ECDSA(hashes.SHA256())))
vr_m, vr_s, _ = benchmark(lambda: ec_pub.verify(ec_sig, message, ec.ECDSA(hashes.SHA256())))
results["ECDSA-P256"] = {
    "type": "SIG",
    "keygen_ms": kg_m, "keygen_std": kg_s,
    "op1_label": "sign_ms",   "op1_ms": sn_m, "op1_std": sn_s,
    "op2_label": "verify_ms", "op2_ms": vr_m, "op2_std": vr_s,
    "pub_key_bytes": len(ec_pub_bytes), "ct_sig_bytes": len(ec_sig)
}

# Save
with open("pqc_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n=== DONE === Results saved to pqc_results.json")
print(f"\n{'Algorithm':<28} {'KeyGen':>10} {'Sign/Encap':>12} {'Verify/Decap':>14} {'PubKey(B)':>10} {'CT/Sig(B)':>10}")
print("-"*90)
for alg, r in results.items():
    print(f"{alg:<28} {r['keygen_ms']:>10.4f} {r['op1_ms']:>12.4f} {r['op2_ms']:>14.4f} {r['pub_key_bytes']:>10} {r['ct_sig_bytes']:>10}")

In [ ]:
import platform, sys, cryptography, psutil

print("Python:", sys.version)
print("OS:", platform.platform())
print("CPU:", platform.processor())
print("CPU cores:", psutil.cpu_count(logical=False), "physical /", psutil.cpu_count(), "logical")
print("RAM:", round(psutil.virtual_memory().total / 1e9, 1), "GB")
print("liboqs version:", oqs.oqs_version())
print("liboqs-python version:", oqs.oqs_python_version())
print("cryptography version:", cryptography.__version__)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

#DATA
labels  = ["ML-KEM-512","ML-KEM-768","ML-KEM-1024",
           "ML-DSA-44","ML-DSA-65","ML-DSA-87",
           "SLH-DSA\n128s","SLH-DSA\n192s","SLH-DSA\n256s",
           "RSA-2048","ECDSA\nP-256"]
keygen  = [0.0192,0.0256,0.0334,0.0516,0.0830,0.1226,123.85,176.53,116.87,76.705,0.0240]
op1     = [0.0193,0.0313,0.0429,0.1327,0.2108,0.2577,933.61,1672.19,1486.67,0.8283,0.0387]
op2     = [0.0198,0.0294,0.0393,0.0487,0.0770,0.1250,1.3954,1.3370,1.8661,0.0310,0.0959]
pubkey  = [800,1184,1568,1312,1952,2592,32,48,64,294,91]
ct_sig  = [768,1088,1568,2420,3309,4627,7856,16224,29792,256,70]

colors  = ["#4C72B0","#4C72B0","#4C72B0",
           "#DD8452","#DD8452","#DD8452",
           "#55A868","#55A868","#55A868",
           "#C44E52","#8172B2"]
legend_patches = [
    mpatches.Patch(color="#4C72B0", label="ML-KEM (FIPS 203)"),
    mpatches.Patch(color="#DD8452", label="ML-DSA (FIPS 204)"),
    mpatches.Patch(color="#55A868", label="SLH-DSA (FIPS 205)"),
    mpatches.Patch(color="#C44E52", label="RSA-2048"),
    mpatches.Patch(color="#8172B2", label="ECDSA P-256"),
]
x = np.arange(len(labels))

# ── FIG 1: Key Generation Time (log scale) ────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x, keygen, color=colors, width=0.6, edgecolor="white")
ax.set_yscale("log")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("Time (ms, log scale)", fontsize=12)
ax.set_xlabel("Algorithm", fontsize=12)
ax.set_title("Fig. 1 — Key Generation Latency (log scale)\nliboqs 0.15.0 · Google Colab x86-64 · 1,000 iterations", fontsize=13)
ax.legend(handles=legend_patches, loc="upper right", fontsize=9)
ax.yaxis.grid(True, which="both", linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("fig1_keygen.png", dpi=300)
plt.show()
print("Saved fig1_keygen.png")

# ── FIG 2: Sign/Encap vs Verify/Decap (two subplots) ─────────────────────────
non_slh_i = [i for i,l in enumerate(labels) if "SLH" not in l]
slh_i     = [i for i,l in enumerate(labels) if "SLH" in l]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios":[2,1]})
w = 0.35
for ax, idxs, title in [(ax1, non_slh_i, "ML-KEM, ML-DSA & Classical"),
                         (ax2, slh_i,     "SLH-DSA (separate scale)")]:
    xi = np.arange(len(idxs))
    ax.bar(xi - w/2, [op1[i] for i in idxs], width=w,
           color=[colors[i] for i in idxs], label="Encap / Sign", edgecolor="white")
    ax.bar(xi + w/2, [op2[i] for i in idxs], width=w,
           color=[colors[i] for i in idxs], alpha=0.45, label="Decap / Verify", edgecolor="white")
    ax.set_xticks(xi)
    ax.set_xticklabels([labels[i] for i in idxs], fontsize=9)
    ax.set_ylabel("Time (ms)", fontsize=11)
    ax.set_xlabel("Algorithm", fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)

# shared legend
solid = mpatches.Patch(color="#888", label="Encap / Sign (solid)")
faded = mpatches.Patch(color="#888", alpha=0.45, label="Decap / Verify (faded)")
fig.legend(handles=[solid, faded], loc="upper center", ncol=2, fontsize=10, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Fig. 2 — Encap/Sign vs Decap/Verify Latency (ms) · 1,000 iterations", fontsize=13, y=1.05)
plt.tight_layout()
plt.savefig("fig2_ops.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved fig2_ops.png")

# ── FIG 3: Sizes (log scale) ──────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, ylabel, title in [
    (ax1, pubkey, "Public Key Size (bytes)", "Public Key Size"),
    (ax2, ct_sig, "Ciphertext / Signature Size (bytes)", "Ciphertext / Signature Size")]:
    ax.bar(x, data, color=colors, width=0.6, edgecolor="white")
    ax.set_yscale("log")
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xlabel("Algorithm", fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.yaxis.grid(True, which="both", linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)

ax1.legend(handles=legend_patches, fontsize=9)
fig.suptitle("Fig. 3 — Public Key & Output Sizes by Algorithm (log scale)", fontsize=13)
plt.tight_layout()
plt.savefig("fig3_sizes.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved fig3_sizes.png")

# ── FIG 4: Scaling across security levels ────────────────────────────────────
levels = [1, 3, 5]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for ax, series, title in [
    (ax1, [
        ("ML-KEM KeyGen",  [0.0192,0.0256,0.0334], "#4C72B0", "-",  "o"),
        ("ML-KEM Encap",   [0.0193,0.0313,0.0429], "#4C72B0", "--", "s"),
        ("ML-DSA KeyGen",  [0.0516,0.0830,0.1226], "#DD8452", "-",  "o"),
        ("ML-DSA Sign",    [0.1327,0.2108,0.2577], "#DD8452", "--", "s"),
    ], "ML-KEM & ML-DSA"),
    (ax2, [
        ("SLH-DSA KeyGen", [123.85,176.53,116.87],    "#55A868", "-",  "o"),
        ("SLH-DSA Sign",   [933.61,1672.19,1486.67],  "#55A868", "--", "s"),
    ], "SLH-DSA (separate scale)")]:
    for name, vals, color, ls, marker in series:
        ax.plot(levels, vals, color=color, linestyle=ls,
                marker=marker, markersize=7, linewidth=2, label=name)
    ax.set_xticks(levels); ax.set_xticklabels(["L1","L3","L5"], fontsize=11)
    ax.set_xlabel("Security Level", fontsize=11)
    ax.set_ylabel("Time (ms)", fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)

fig.suptitle("Fig. 4 — Latency Scaling Across NIST Security Levels (1, 3, 5)", fontsize=13)
plt.tight_layout()
plt.savefig("fig4_scaling.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved fig4_scaling.png")